# Notebook 2 — `training_normalizer.ipynb`

**Sovereign Dialect-Bridge · Step 2 — Train mT5-small Normalizer (Stage 1)**

Fine-tune **mT5-small** untuk menerjemahkan kalimat dialek (Jawa/Sunda/Minang/dll.)
ke Bahasa Indonesia baku. Output disimpan di `models/normalizer/` (dan Google Drive
jika berjalan di Colab agar tidak hilang saat session restart).

| Sumber | Volume | Kualitas | Dialek |
|--------|--------|----------|--------|
| `GEM/indonlg` (mt_id_jv, mt_id_su) | 5–10K/dialek | human-translated, EMNLP 2021 | Jawa, Sunda |
| `Exqrch/IndonesianNMT` (id_jav, id_sun, id_min) | max 30K/dialek | besar | Jawa, Sunda, Minang |
| `indonlp/NusaX-MT` | ~1K total | native speaker | 10 dialek lain |

**Target:** BLEU-4 > 20 pada validation set.
**Hardware:** Google Colab T4 (16 GB VRAM, fp32) atau RTX 3090/4090 (bf16, lebih cepat).


In [ ]:
import sys, importlib, subprocess

PACKAGES = {
    "transformers"   : "transformers==4.40.0",
    "datasets"       : "datasets",
    "accelerate"     : "accelerate",
    "sentencepiece"  : "sentencepiece",
    "tokenizers"     : "tokenizers",
    "google.protobuf": "protobuf",
    "sacrebleu"      : "sacrebleu",
    "sacremoses"     : "sacremoses",
    "sklearn"        : "scikit-learn",
    "pandas"         : "pandas",
    "pyarrow"        : "pyarrow",
    "numpy"          : "numpy",
    "matplotlib"     : "matplotlib",
}

to_install = []
for module, pkg in PACKAGES.items():
    try:
        importlib.import_module(module)
        print(f"  [ok]      {pkg}")
    except ImportError:
        print(f"  [missing] {pkg}")
        to_install.append(pkg)

if to_install:
    print(f"\nInstalling {len(to_install)} package(s)...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + to_install,
        check=True,
    )
    print("Selesai. Lanjutkan run ke bawah.")
else:
    print("\nSemua package sudah terinstall.")


## 1. Setup environment

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]   = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = torch.cuda.is_bf16_supported() if DEVICE == "cuda" else False
USE_FP16 = False  # JANGAN fp16 — NaN crash di mT5

assert DEVICE == "cuda", (
    "Notebook butuh GPU. Di Colab: Runtime > Change runtime type > T4 GPU."
)
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB  |  bf16: {USE_BF16}")
if not USE_BF16:
    print("T4 detected — training dalam fp32 (bf16 tidak tersedia di Turing). Normal dan aman untuk mT5.")
print(f"torch: {torch.__version__}")


In [ ]:
# Mount Google Drive agar model tidak hilang saat Colab session restart
SAVE_TO_DRIVE = False
DRIVE_ROOT    = None

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT    = Path("/content/drive/MyDrive/sovereign-dialect-bridge")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    SAVE_TO_DRIVE = True
    print(f"Google Drive mounted. Persistent path: {DRIVE_ROOT}")
except ModuleNotFoundError:
    print("Bukan Colab — model disimpan lokal saja (vast.ai / laptop).")


## 2. Global configuration

In [ ]:
# Auto-detect project root
CWD = Path.cwd()
if (CWD / "dataset").exists():
    ROOT = CWD
elif (CWD.parent / "dataset").exists():
    ROOT = CWD.parent
else:
    ROOT = CWD  # Colab: /content, atau workspace lain

MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR   = ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Model
NORM_MODEL      = "google/mt5-small"
NORM_MAX_INPUT  = 256
NORM_MAX_TARGET = 256
NORM_OUTPUT_DIR = str(MODELS_DIR / "normalizer")

# Hyperparameter — disesuaikan dengan hardware
# T4 (fp32): batch=4, grad_accum=4 → eff_batch=16
# RTX 3090 (bf16): batch=8, grad_accum=2 → eff_batch=16 (sama)
NORM_LR             = 5e-5   # KRITIS: jangan naikkan, mT5 NaN crash jika lebih tinggi
NORM_EPOCHS         = 5      # 5 epoch untuk MT task, lebih dari summarization
NORM_BATCH          = 4 if not USE_BF16 else 8   # T4 fp32: 4; RTX 30xx bf16: 8
NORM_GRAD_ACCUM     = 4 if not USE_BF16 else 2   # eff_batch selalu 16
NORM_MAX_GRAD_NORM  = 1.0    # gradient clipping
LABEL_SMOOTHING     = 0.1    # regularisasi distribusi output
WARMUP_RATIO        = 0.1    # 10% step pertama sebagai warmup
WEIGHT_DECAY        = 0.01   # L2 regularisasi
EARLY_STOP_PATIENCE = 2      # hentikan jika eval_loss stagnan 2 epoch
EVAL_ACCUM_STEPS    = 4      # akumulasi step saat eval, cegah OOM

# Cap IndonesianNMT — Colab RAM ~12 GB; IndonesianNMT bisa 100K+ per dialek
MAX_NMT_PER_DIALECT = 30000

RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)

print(f"ROOT         : {ROOT}")
print(f"NORM_OUTPUT  : {NORM_OUTPUT_DIR}")
print(f"LR={NORM_LR}  batch={NORM_BATCH}  grad_accum={NORM_GRAD_ACCUM}  eff_batch={NORM_BATCH * NORM_GRAD_ACCUM}")
print(f"bf16={USE_BF16}  early_stop_patience={EARLY_STOP_PATIENCE}  max_nmt_per_dialect={MAX_NMT_PER_DIALECT:,}")


## 3. Load datasets

Tiga sumber digabung secara berurutan berdasarkan prioritas kualitas.
Setiap sumber di-load dalam sel terpisah agar jika satu gagal, yang lain tetap jalan.

In [ ]:
from datasets import load_dataset

mt_data = []

# Sumber 1: IndoNLG MT — prioritas tertinggi (human-translated, EMNLP 2021 benchmark)
INDONLG_CONFIGS = [("mt_id_jv", "javanese"), ("mt_id_su", "sundanese")]
for cfg, dialect in INDONLG_CONFIGS:
    try:
        ds       = load_dataset("GEM/indonlg", cfg, trust_remote_code=True)
        n_before = len(mt_data)
        for split in ds.keys():
            for row in ds[split]:
                bi   = str(row.get("target") or "")
                refs = row.get("references") or []
                dial = str(refs[0]) if refs else ""
                if bi and dial:
                    mt_data.append({"indonesian": bi, "dialect": dial,
                                    "dialect_name": dialect, "source": "indonlg"})
        print(f"  [ok] IndoNLG {cfg:10s} : {len(mt_data) - n_before:,} pairs")
    except Exception as e:
        print(f"  [fail] IndoNLG {cfg:10s} : {e}")

print(f"Subtotal setelah IndoNLG: {len(mt_data):,} pairs")


In [ ]:
# Sumber 2: IndonesianNMT — volume besar, di-cap agar tidak OOM di Colab
NMT_SUBSETS = [("id_jav", "javanese"), ("id_sun", "sundanese"), ("id_min", "minangkabau")]
for subset, dialect in NMT_SUBSETS:
    try:
        ds       = load_dataset("Exqrch/IndonesianNMT", subset, trust_remote_code=True)
        n_before = len(mt_data)
        n_added  = 0
        for split in ds.keys():
            if n_added >= MAX_NMT_PER_DIALECT:
                break
            for row in ds[split]:
                if n_added >= MAX_NMT_PER_DIALECT:
                    break
                cols     = list(row.keys())
                bi_col   = next((c for c in cols if c.lower() in
                                 {"ind","id","indonesian","ind_id","idn"}), None)
                dial_col = next((c for c in cols if c != bi_col), None)
                if bi_col and dial_col:
                    bi = str(row[bi_col] or "")
                    dl = str(row[dial_col] or "")
                    if bi and dl:
                        mt_data.append({"indonesian": bi, "dialect": dl,
                                        "dialect_name": dialect, "source": "indonesian_nmt"})
                        n_added += 1
        print(f"  [ok] IndonesianNMT {subset:8s} : {len(mt_data) - n_before:,} pairs "
              f"(cap={MAX_NMT_PER_DIALECT:,})")
    except Exception as e:
        print(f"  [fail] IndonesianNMT {subset:8s} : {e}")

print(f"Subtotal setelah IndonesianNMT: {len(mt_data):,} pairs")


In [ ]:
# Sumber 3: NusaX-MT — dialek lain (backup), juga untuk evaluasi & dialect dict
df_nusax    = None
DIALECT_COLS = []

try:
    nusax       = load_dataset("indonlp/NusaX-MT", trust_remote_code=True)
    train_split = nusax.get("train") or nusax[list(nusax.keys())[0]]
    cols        = train_split.column_names
    bi_col      = next((c for c in cols if c.lower() in {"indonesian","ind","id"}), None)
    DIALECT_COLS = [c for c in cols if c not in (bi_col, "english", "id", "gem_id", "index")]
    n_before    = len(mt_data)
    for split_name in nusax.keys():
        for row in nusax[split_name]:
            bi = str(row.get(bi_col) or "")
            for d in DIALECT_COLS:
                dial = str(row.get(d) or "")
                if bi and dial:
                    mt_data.append({"indonesian": bi, "dialect": dial,
                                    "dialect_name": d, "source": "nusax"})
    print(f"  [ok] NusaX-MT HF : {len(mt_data) - n_before:,} pairs "
          f"across {len(DIALECT_COLS)} dialects")
    df_nusax = (nusax["train"].to_pandas() if "train" in nusax
                else train_split.to_pandas())

except Exception as e:
    print(f"  [fail] NusaX-MT HF : {e}")
    print("  Mencoba CSV lokal...")
    local = ROOT / "dataset" / "nusax" / "datasets" / "mt"
    if local.exists():
        dfs = [pd.read_csv(local / f)
               for f in ["train.csv", "valid.csv", "test.csv"]
               if (local / f).exists()]
        if dfs:
            df_nusax = pd.concat(dfs, ignore_index=True)
            cols     = df_nusax.columns.tolist()
            bi_col   = next((c for c in cols if c.lower() in
                             {"indonesian","ind","id"}), "indonesian")
            DIALECT_COLS = [c for c in cols
                            if c not in (bi_col, "english", "Unnamed: 0", "id")]
            n_before = len(mt_data)
            for _, row in df_nusax.iterrows():
                bi = str(row.get(bi_col) or "")
                for d in DIALECT_COLS:
                    dial = str(row.get(d) or "")
                    if bi and dial:
                        mt_data.append({"indonesian": bi, "dialect": dial,
                                        "dialect_name": d, "source": "nusax_local"})
            print(f"  [ok] NusaX lokal : {len(mt_data) - n_before:,} pairs")
    else:
        print("  [fail] NusaX lokal tidak ditemukan. Lanjut tanpa NusaX.")

print(f"\nTOTAL MT pairs terkumpul: {len(mt_data):,}")


## 4. Filter & split

In [ ]:
df_mt = pd.DataFrame(mt_data).dropna(subset=["indonesian", "dialect"])
df_mt = df_mt[
    df_mt["indonesian"].str.strip().astype(bool) &
    df_mt["dialect"].str.strip().astype(bool)
]
# Buang pasangan yang terlalu pendek (< 3 kata)
df_mt = df_mt[
    (df_mt["dialect"].str.split().str.len()    >= 3) &
    (df_mt["indonesian"].str.split().str.len() >= 3)
]
df_mt = df_mt.drop_duplicates(subset=["indonesian", "dialect"]).reset_index(drop=True)

print(f"Filtered: {len(df_mt):,} pairs")
print(f"\nPer dialect:")
print(df_mt["dialect_name"].value_counts().to_string())
print(f"\nPer source:")
print(df_mt["source"].value_counts().to_string())


In [ ]:
from sklearn.model_selection import train_test_split

strat = df_mt["dialect_name"] if df_mt["dialect_name"].nunique() > 1 else None
df_train_mt, df_val_mt = train_test_split(
    df_mt, test_size=0.05, random_state=RANDOM_SEED, stratify=strat
)
df_train_mt = df_train_mt.reset_index(drop=True)
df_val_mt   = df_val_mt.reset_index(drop=True)

print(f"Train : {len(df_train_mt):,}  |  Val: {len(df_val_mt):,}")
print(f"\nContoh pasangan training:")
for i in range(min(3, len(df_train_mt))):
    row = df_train_mt.iloc[i]
    print(f"  [{row['dialect_name']}]")
    print(f"    DIALECT : {row['dialect'][:75]}")
    print(f"    BI      : {row['indonesian'][:75]}")
    print()


## 5. Tokenization

In [ ]:
from transformers import MT5Tokenizer

tokenizer = MT5Tokenizer.from_pretrained(NORM_MODEL)
print(f"Tokenizer: {NORM_MODEL}  vocab={len(tokenizer):,}")


def tokenize_pair(example):
    inputs = tokenizer(
        example["dialect"],
        max_length=NORM_MAX_INPUT,
        truncation=True,
    )
    labels = tokenizer(
        text_target=example["indonesian"],
        max_length=NORM_MAX_TARGET,
        truncation=True,
    )
    # Ganti pad_token_id dengan -100 agar padding tidak dihitung dalam loss
    pad = tokenizer.pad_token_id
    inputs["labels"] = [
        (tok if tok != pad else -100)
        for tok in labels["input_ids"]
    ]
    return inputs


In [ ]:
from datasets import Dataset

ds_train = Dataset.from_pandas(df_train_mt[["dialect", "indonesian"]].reset_index(drop=True))
ds_val   = Dataset.from_pandas(df_val_mt[["dialect",   "indonesian"]].reset_index(drop=True))

ds_train = ds_train.map(tokenize_pair, batched=False, remove_columns=ds_train.column_names)
ds_val   = ds_val.map(tokenize_pair,   batched=False, remove_columns=ds_val.column_names)

print(f"Tokenized train : {len(ds_train):,}")
print(f"Tokenized val   : {len(ds_val):,}")
print(f"Sample input_ids length : {len(ds_train[0]['input_ids'])}")
print(f"Sample labels length    : {len(ds_train[0]['labels'])}")
print(f"Labels contain -100: {-100 in ds_train[0]['labels']}")


## 6. Load model + DataCollator

In [ ]:
from transformers import MT5ForConditionalGeneration, DataCollatorForSeq2Seq

model = MT5ForConditionalGeneration.from_pretrained(NORM_MODEL)
model.config.decoder_start_token_id = tokenizer.pad_token_id

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model: {NORM_MODEL}  ({n_params:.1f}M params)")
if DEVICE == "cuda":
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM setelah load model: {(total - free)/1e9:.1f} GB used / {total/1e9:.1f} GB total")

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model,
    padding="longest", label_pad_token_id=-100, return_tensors="pt",
)


## 7. Training configuration

In [ ]:
# Konfigurasi training — setiap parameter ada alasannya
NORM_CONFIG = {
    "num_train_epochs"            : NORM_EPOCHS,           # 5 epoch cukup untuk MT task
    "per_device_train_batch_size" : NORM_BATCH,            # T4: 4; RTX 30xx: 8
    "per_device_eval_batch_size"  : NORM_BATCH,            # sama dengan train
    "gradient_accumulation_steps" : NORM_GRAD_ACCUM,       # eff_batch = batch * grad_accum = 16
    "learning_rate"               : NORM_LR,               # 5e-5, KRITIS untuk mT5
    "warmup_ratio"                : WARMUP_RATIO,          # 10% langkah pertama sebagai warmup
    "weight_decay"                : WEIGHT_DECAY,          # L2 regularisasi
    "max_grad_norm"               : NORM_MAX_GRAD_NORM,    # gradient clipping 1.0
    "label_smoothing_factor"      : LABEL_SMOOTHING,       # 0.1 regularisasi distribusi output
    "bf16"                        : USE_BF16,              # True di RTX 30xx/40xx, False di T4
    "fp16"                        : False,                 # JANGAN: NaN overflow di mT5
    "gradient_checkpointing"      : True,                  # recompute aktivasi, hemat VRAM
    "eval_accumulation_steps"     : EVAL_ACCUM_STEPS,      # akumulasi langkah eval, cegah OOM
    "early_stopping_patience"     : EARLY_STOP_PATIENCE,   # berhenti jika 2 epoch stagnan
}
print("Normalizer config:")
for k, v in NORM_CONFIG.items():
    print(f"  {k:35s}: {v}")


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback

args = Seq2SeqTrainingArguments(
    output_dir                  = NORM_OUTPUT_DIR,
    num_train_epochs            = NORM_CONFIG["num_train_epochs"],
    per_device_train_batch_size = NORM_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size  = NORM_CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps = NORM_CONFIG["gradient_accumulation_steps"],
    learning_rate               = NORM_CONFIG["learning_rate"],
    warmup_ratio                = NORM_CONFIG["warmup_ratio"],
    weight_decay                = NORM_CONFIG["weight_decay"],
    max_grad_norm               = NORM_CONFIG["max_grad_norm"],
    label_smoothing_factor      = NORM_CONFIG["label_smoothing_factor"],
    bf16                        = NORM_CONFIG["bf16"],
    fp16                        = NORM_CONFIG["fp16"],
    gradient_checkpointing      = NORM_CONFIG["gradient_checkpointing"],
    eval_accumulation_steps     = NORM_CONFIG["eval_accumulation_steps"],
    predict_with_generate       = True,
    generation_max_length       = NORM_MAX_TARGET,
    generation_num_beams        = 4,
    evaluation_strategy         = "epoch",   # transformers 4.40.0 — jangan pakai eval_strategy
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    save_total_limit            = 2,
    logging_steps               = 100,
    report_to                   = "none",
    dataloader_pin_memory       = False,
    seed                        = RANDOM_SEED,
)

trainer = Seq2SeqTrainer(
    model         = model,
    args          = args,
    train_dataset = ds_train,
    eval_dataset  = ds_val,
    data_collator = collator,
    tokenizer     = tokenizer,
    callbacks     = [EarlyStoppingCallback(
        early_stopping_patience=NORM_CONFIG["early_stopping_patience"]
    )],
)
n_steps = (len(ds_train) // (NORM_BATCH * NORM_GRAD_ACCUM)) * NORM_EPOCHS
print(f"Trainer ready.")
print(f"Estimasi steps: ~{n_steps:,} ({NORM_EPOCHS} epoch x {len(ds_train):,} / {NORM_BATCH * NORM_GRAD_ACCUM})")


## 8. Train

In [ ]:
NORM_TRAINED = False
try:
    model.to(DEVICE)
    print(f"Mulai training mT5-small normalizer")
    print(f"  Data  : {len(ds_train):,} train  |  {len(ds_val):,} val")
    print(f"  Epoch : {NORM_EPOCHS}  |  eff. batch: {NORM_BATCH * NORM_GRAD_ACCUM}")
    trainer.train()
    trainer.save_model(NORM_OUTPUT_DIR)
    tokenizer.save_pretrained(NORM_OUTPUT_DIR)
    NORM_TRAINED = True
    print(f"\nSaved normalizer -> {NORM_OUTPUT_DIR}")
except Exception as _e:
    print(f"[WARNING] Training gagal: {type(_e).__name__}: {_e}")
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


## 9. Loss curves

In [ ]:
if NORM_TRAINED:
    log        = trainer.state.log_history
    tr_steps   = [x["step"]      for x in log if "loss"      in x and "eval_loss" not in x]
    tr_losses  = [x["loss"]      for x in log if "loss"      in x and "eval_loss" not in x]
    val_steps  = [x["step"]      for x in log if "eval_loss" in x]
    val_losses = [x["eval_loss"] for x in log if "eval_loss" in x]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(tr_steps,  tr_losses,  color="steelblue", label="Train Loss")
    ax.plot(val_steps, val_losses, color="coral",     label="Val Loss", marker="o")
    ax.set_title("mT5-small Normalizer — Training & Validation Loss")
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.legend()
    plt.tight_layout()
    plt.show()
    if tr_losses:
        print(f"Final train loss : {tr_losses[-1]:.4f}")
    if val_losses:
        print(f"Final val loss   : {val_losses[-1]:.4f}")
    stopped = trainer.state.global_step
    total   = trainer.state.max_steps
    if stopped < total:
        print(f"Early stopping aktif: berhenti di step {stopped} dari {total}")
else:
    print("Training tidak berhasil — skip loss curves.")


## 10. Evaluasi — BLEU-4 + chrF++

Target: **BLEU-4 > 20** sebelum normalizer dipakai di pipeline inferensi.

In [ ]:
import sacrebleu


def evaluate_normalizer(model, tokenizer, val_df, n=200):
    model.eval()
    preds, refs = [], []
    sample = val_df.sample(min(n, len(val_df)), random_state=RANDOM_SEED)
    for _, row in sample.iterrows():
        inputs = tokenizer(
            row["dialect"], return_tensors="pt",
            max_length=NORM_MAX_INPUT, truncation=True
        ).to(DEVICE)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens       = NORM_MAX_TARGET,
                num_beams            = 4,
                no_repeat_ngram_size = 3,
                early_stopping       = True,
            )
        preds.append(tokenizer.decode(out[0], skip_special_tokens=True))
        refs.append(row["indonesian"])

    bleu = sacrebleu.corpus_bleu(preds, [refs])
    chrf = sacrebleu.corpus_chrf(preds, [refs])
    print(f"BLEU-4  : {bleu.score:.2f}  (target: > 20)")
    print(f"chrF++  : {chrf.score:.2f}")
    return {
        "bleu4": round(bleu.score, 2),
        "chrf" : round(chrf.score, 2),
        "preds": preds,
        "refs" : refs,
    }


if NORM_TRAINED:
    eval_results = evaluate_normalizer(model, tokenizer, df_val_mt, n=200)
else:
    print("[SKIP] Training tidak berhasil.")
    eval_results = {"bleu4": 0.0, "chrf": 0.0, "preds": [], "refs": []}


In [ ]:
if eval_results["preds"]:
    print("Contoh prediksi normalizer (dialect -> BI baku):\n")
    for i in range(min(5, len(eval_results["preds"]))):
        row = df_val_mt.iloc[i]
        print(f"[{row['dialect_name']}]")
        print(f"  DIALECT : {row['dialect'][:80]}")
        print(f"  PRED    : {eval_results['preds'][i][:80]}")
        print(f"  REF     : {eval_results['refs'][i][:80]}")
        print()


## 11. Auto-expand dialect dictionary

Ekstrak pasangan kata dari NusaX untuk kamus rule-based.
Disimpan ke `data/dialect_dict.json`, digunakan oleh notebook 4 sebagai fallback.

In [ ]:
def extract_dialect_pairs(df, dialect_col, bi_col="indonesian", top_n=300):
    pairs = {}
    if dialect_col not in df.columns:
        return pairs
    for _, row in df.iterrows():
        bi   = str(row.get(bi_col,      "") or "").lower().split()
        dial = str(row.get(dialect_col, "") or "").lower().split()
        if len(bi) != len(dial):
            continue
        for d, b in zip(dial, bi):
            if d != b and len(d) > 2 and len(b) > 2 and d.isalpha() and b.isalpha():
                pairs.setdefault(d, b)
        if len(pairs) >= top_n:
            break
    return pairs


DIALECT_TARGET_COLS = [
    "javanese", "sundanese", "minangkabau", "balinese", "banjarese",
    "madurese", "acehnese", "buginese", "ngaju", "toba_batak",
]

dialect_dict = {}
if df_nusax is not None:
    for d in DIALECT_TARGET_COLS:
        if d in df_nusax.columns:
            pairs = extract_dialect_pairs(df_nusax, d, top_n=300)
            dialect_dict[d] = pairs
            print(f"  {d:14s} : {len(pairs):3d} pairs")
else:
    print("df_nusax tidak tersedia — dialect_dict kosong.")

dict_path = DATA_DIR / "dialect_dict.json"
with open(dict_path, "w", encoding="utf-8") as f:
    json.dump(dialect_dict, f, ensure_ascii=False, indent=2)
n_total = sum(len(v) for v in dialect_dict.values())
print(f"\nSaved: {dict_path}  ({n_total} total pairs)")


## 12. Simpan ke Google Drive & bersihkan VRAM

Model disalin ke Google Drive agar tidak hilang saat Colab session berakhir.

In [ ]:
import shutil

if NORM_TRAINED:
    if SAVE_TO_DRIVE and DRIVE_ROOT is not None:
        drive_norm_dir = DRIVE_ROOT / "models" / "normalizer"
        if drive_norm_dir.exists():
            shutil.rmtree(drive_norm_dir)
        shutil.copytree(NORM_OUTPUT_DIR, str(drive_norm_dir))
        print(f"Model disalin ke Google Drive: {drive_norm_dir}")

        # Salin dialect_dict.json juga
        drive_data_dir = DRIVE_ROOT / "data"
        drive_data_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(DATA_DIR / "dialect_dict.json"),
                     str(drive_data_dir / "dialect_dict.json"))
        print(f"dialect_dict.json disalin ke Drive.")
    else:
        print(f"Model tersimpan di: {NORM_OUTPUT_DIR}")
else:
    print("[SKIP] Training tidak berhasil.")

# Bebaskan VRAM
del model, trainer, collator, tokenizer
gc.collect()
torch.cuda.empty_cache()
if DEVICE == "cuda":
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM bebas: {free/1e9:.1f} GB / {total/1e9:.1f} GB")


## Selesai

Output yang dihasilkan:
- `models/normalizer/` — mT5-small fine-tuned + tokenizer (local)
- `data/dialect_dict.json` — kamus dialek hasil auto-expand dari NusaX
- Google Drive: `MyDrive/sovereign-dialect-bridge/models/normalizer/` (jika di Colab)

**Langkah selanjutnya:** jalankan `training_sum.ipynb` untuk training Stage 2 (IndoBART, mT5-base, IndoT5).
